# Configuring Claude Code for a Multi-Developer Project

**Hands-on exercise — bottom-up walkthrough**

This notebook completes the CCAF Foundations exercise on team-scale Claude Code configuration. You build a small sample project from the ground up and, layer by layer, add every configuration mechanism a team relies on: a project `CLAUDE.md`, path-scoped rules, a forked skill, MCP server config, and a plan-mode decision check.

**Bottom-up approach:** each section creates a real file inside a throwaway sandbox project, then verifies it. By the end you have a fully configured project you can inspect.

### Agenda
1. Project-level `CLAUDE.md` — universal standards every teammate inherits
2. `.claude/rules/` — path-scoped conventions that load conditionally
3. `.claude/skills/` + `.claude/commands/` — forked skills and slash commands
4. `.mcp.json` + `~/.claude.json` — project vs personal MCP servers
5. Plan mode vs direct execution — choosing the right mode per task

> A note on scope: this notebook configures **Claude Code** (the CLI / harness), not the Claude API. The code cells write and inspect configuration files — they do not call the Anthropic API, so no API key is needed. Everything is created inside `ex1/team_project/` so your real repo and home directory stay untouched.

## 0. Setup (do this before class starts)

No API key, no installs — this exercise touches only the filesystem.

1. Requirements: Python 3.10+. No extra packages.
2. The next code cell creates a sandbox project at `ex1/team_project/` (relative to this notebook) and defines three helpers used throughout:
   - `write_file(relpath, content)` — writes a file (creating parent dirs) inside the sandbox
   - `show_tree()` — prints the sandbox directory tree
   - `cat(relpath)` — prints a file's contents with a header
3. Re-running the setup cell is safe: it wipes and recreates the sandbox, so every run starts clean.

**Why a sandbox:** Step 4 involves `~/.claude.json`. We never write to your real home directory — instead we simulate the user-level config inside the sandbox so the exercise is reproducible and fully reversible.

In [ ]:
import shutil
from pathlib import Path

# Sandbox project root — everything in this notebook lives here.
PROJECT = Path("ex1/team_project").resolve()

# Start clean on every run.
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)


def write_file(relpath, content):
    """Write content to PROJECT/relpath, creating parent directories."""
    path = PROJECT / relpath
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.lstrip("\n"))
    print(f"wrote  {relpath}  ({len(content.strip().splitlines())} lines)")
    return path


def show_tree(root=None):
    """Print the sandbox directory tree."""
    root = root or PROJECT
    print(f"{root.name}/")
    for entry in sorted(root.rglob("*")):
        rel = entry.relative_to(root)
        depth = len(rel.parts) - 1
        marker = "/" if entry.is_dir() else ""
        print("  " * (depth + 1) + rel.parts[-1] + marker)


def cat(relpath):
    """Print a file's contents with a header."""
    path = PROJECT / relpath
    print(f"\n===== {relpath} " + "=" * max(0, 52 - len(str(relpath))))
    print(path.read_text())


print("Python OK")
print(f"Sandbox: {PROJECT}")
print(f"Sandbox ready: {PROJECT.is_dir()}")

---
# 1. Project-level CLAUDE.md — the shared baseline

`CLAUDE.md` is the file Claude Code loads automatically as project memory. Its job: encode the standards every teammate should get for free, without anyone re-typing them.

### The configuration hierarchy

| Level | Location | Shared with team? | Use for |
|---|---|---|---|
| User | `~/.claude/CLAUDE.md` | No — personal, not in version control | Your own preferences |
| Project | `CLAUDE.md` (repo root) or `.claude/CLAUDE.md` | Yes — committed to git | Team standards |
| Directory | `CLAUDE.md` inside a subfolder | Yes | Standards for one package |

All three layers stack: Claude reads the user file, then the project file, then any directory `CLAUDE.md` covering the files in play.

### A common failure mode

A teammate reports "Claude isn't following our test conventions." The usual cause: the conventions live in someone's **user-level** `~/.claude/CLAUDE.md`, which never left their machine. Fix: move shared standards to the **project** `CLAUDE.md` so version control distributes them. Use `/memory` to see exactly which memory files a session loaded.

### @import for modularity

`CLAUDE.md` can pull in other files with `@path/to/file.md`. This keeps the root file short while letting each package import only the standards it needs:

```
@.claude/standards/python.md
@.claude/standards/git.md
```

### CLAUDE.md vs a skill

`CLAUDE.md` is **always loaded** — use it for universal, short standards. A skill is **invoked on demand** — use it for task-specific workflows. Putting a rarely needed 200-line workflow in `CLAUDE.md` taxes every request's context.

### Demo: write the project CLAUDE.md and a directory-level override

The cell writes a root `CLAUDE.md` with universal standards (style, testing conventions) plus one `@import`, the imported `git.md` standards file, and a **directory-level** `CLAUDE.md` inside `src/api/` that adds a rule specific to that package. It then prints the tree and both `CLAUDE.md` files. Watch how the directory file is short — it assumes the root file already applies and only adds what is local. Swap a rule in the root file to see what every teammate would inherit.

In [ ]:
write_file("CLAUDE.md", """
# Team Project — Claude Code instructions

Universal standards. Committed to git, so every teammate's Claude Code loads this.

## Coding standards
- Python 3.10+. Format with `ruff format`; lint with `ruff check`.
- Functions do one thing. Prefer pure functions; isolate I/O at the edges.
- Type-hint every public function signature.

## Testing conventions
- Every bug fix ships with a regression test that fails before the fix.
- Tests use pytest. One behavior per test; name tests `test_<behavior>`.
- Do not mock what you own — only mock external services.

## Commits
- Conventional Commits: `feat:`, `fix:`, `test:`, `docs:`.

@.claude/standards/git.md
""")

write_file(".claude/standards/git.md", """
# Git workflow (imported into the root CLAUDE.md)

- Branch from `main`; never commit directly to `main`.
- Rebase before opening a PR; keep history linear.
- One PR is one logical change.
""")

write_file("src/api/CLAUDE.md", """
# src/api — directory-level instructions

Inherits the root CLAUDE.md. Only the API-specific additions live here.

- Every endpoint validates its input with a Pydantic model.
- Return typed responses; never return a bare dict.
""")

show_tree()
cat("CLAUDE.md")
cat("src/api/CLAUDE.md")

> **🏫 During class:**
> 1. Run the setup cell, then this cell. Point at the printed tree — three `CLAUDE.md`-related files now exist at three different scopes.
> 2. Say out loud: "The root `CLAUDE.md` is the team contract — it is in git, so a new hire clones the repo and instantly gets every standard. Nothing here depends on someone's laptop."
> 3. Ask the room: "A teammate's tests keep ignoring our naming convention. Where do you look first?" (Answer: their `~/.claude/CLAUDE.md` — a user-level file that is never shared. Move the rule to the project `CLAUDE.md`.) Then have someone run `/memory` in a real session to show which files loaded.

---
# 2. .claude/rules/ — conventions that load only when relevant

A monolithic `CLAUDE.md` has a cost: **every** rule is in context for **every** request, even when irrelevant. API rules do not help while editing a CSS file.

`.claude/rules/` solves this. Each file is one topic, and its YAML frontmatter `paths:` field is a list of glob patterns. The rule loads **only when the file being edited matches a pattern.**

### Anatomy of a rule file

```markdown
---
paths: ["src/api/**/*"]
---
# API conventions
- ...
```

### Glob patterns beat a directory CLAUDE.md

Test files are scattered — `src/api/users.test.js`, `src/ui/button.test.js`, `lib/date.test.js`. A directory-level `CLAUDE.md` is **directory-bound**: you would need one in every folder. A single rule with `paths: ["**/*.test.*"]` covers all of them, regardless of location.

| | Directory CLAUDE.md | .claude/rules/ with globs |
|---|---|---|
| Scope | One directory subtree | Any pattern, cross-cutting |
| Loaded | When editing in that tree | When editing matching files |
| Best for | A package's local rules | A file *type* spread everywhere |

Path-scoped rules also reduce token usage — irrelevant conventions simply never enter context.

### Demo: write two rule files, then simulate which ones load

The cell writes `api-conventions.md` (`paths: ["src/api/**/*"]`) and `testing.md` (`paths: ["**/*.test.*"]`). Then `rules_for(path)` parses each rule's frontmatter and uses a small glob matcher to report which rules Claude Code would load for a given file. Watch the contrast: editing `src/api/users.py` pulls in API rules only; editing `src/api/users.test.py` pulls in **both**; editing `README.md` pulls in none. Change a path in the test list to probe the boundaries.

In [ ]:
import json
import re


def glob_match(path, pattern):
    """Match a POSIX path against a glob pattern supporting ** and *."""
    regex, i = "", 0
    while i < len(pattern):
        if pattern[i:i + 2] == "**":
            regex += ".*"
            i += 2
            if pattern[i:i + 1] == "/":   # let **/ also match zero directories
                i += 1
        elif pattern[i] == "*":
            regex += "[^/]*"
            i += 1
        else:
            regex += re.escape(pattern[i])
            i += 1
    return re.fullmatch(regex, path) is not None


def parse_frontmatter(text):
    """Parse the YAML frontmatter block between the leading --- fences.

    List values (e.g. ["a", "b"]) are valid JSON, so json.loads parses them;
    bare scalars (e.g. fork) fall back to a plain string.
    """
    fm = {}
    if text.startswith("---"):
        block = text[3:text.index("---", 3)]
        for line in block.strip().splitlines():
            if ":" not in line:
                continue
            key, _, val = line.partition(":")
            val = val.strip()
            try:
                fm[key.strip()] = json.loads(val)
            except json.JSONDecodeError:
                fm[key.strip()] = val
    return fm


write_file(".claude/rules/api-conventions.md", """
---
paths: ["src/api/**/*"]
---
# API conventions
- Every endpoint validates input with a Pydantic model.
- Version routes under /v1, /v2; never break a published route.
- Use 4xx for client errors, 5xx only for genuine server faults.
""")

write_file(".claude/rules/testing.md", """
---
paths: ["**/*.test.*"]
---
# Testing conventions
- Arrange-Act-Assert, with a blank line between the three blocks.
- Each test asserts exactly one behavior.
- Name tests after the behavior, not the function under test.
""")

RULES_DIR = PROJECT / ".claude" / "rules"


def rules_for(rel_path):
    """Return the rule files Claude Code loads when editing rel_path."""
    loaded = []
    for rule in sorted(RULES_DIR.glob("*.md")):
        patterns = parse_frontmatter(rule.read_text()).get("paths", [])
        if any(glob_match(rel_path, p) for p in patterns):
            loaded.append(rule.name)
    return loaded


print()
for f in ["src/api/users.py", "src/api/users.test.py", "src/ui/button.css", "README.md"]:
    hits = rules_for(f)
    print(f"editing {f:26s} -> loads: {hits or 'no path-scoped rules'}")

> **🏫 During class:**
> 1. Run the cell. Walk down the four printed lines one at a time.
> 2. Say: "`users.test.py` lives inside `src/api/`, so it matches *both* globs — path rules stack, they are not exclusive. And nothing irrelevant loads for `button.css`; that is tokens saved on every unrelated edit."
> 3. Ask: "Why not just put testing rules in `src/api/CLAUDE.md`?" (Because test files are everywhere — a directory `CLAUDE.md` only covers its own subtree.) Variation: add `"lib/date.test.js"` to the loop and watch the testing rule still fire.

---
# 3. Skills and slash commands — .claude/skills/ and .claude/commands/

Two on-demand mechanisms, both committed to the repo so the whole team gets them.

### Slash commands — .claude/commands/

A Markdown file `.claude/commands/review.md` becomes the `/review` command. Project-scoped commands (`.claude/commands/`) are shared via version control; user-scoped commands (`~/.claude/commands/`) are personal.

### Skills — .claude/skills/

A skill is a folder with a `SKILL.md`. Its frontmatter configures behavior:

| Frontmatter | Effect |
|---|---|
| `context: fork` | Runs the skill in an **isolated sub-agent**; its verbose output never lands in the main conversation — only a summary returns |
| `allowed-tools` | Restricts which tools the skill may use — e.g. a read-only analysis can be denied `Write` and `Bash` |
| `argument-hint` | Prompts the developer for parameters when they invoke the skill with no arguments |

### Why context: fork matters

A "summarize the whole codebase" skill might read 60 files. Without `fork`, all 60 files' contents pile into your main context. With `context: fork` that exploration happens in a sub-agent and you get back just the summary — the main context stays clean.

### Skill vs command vs CLAUDE.md
- **CLAUDE.md** — always loaded, universal, short.
- **Slash command** — a saved prompt you trigger by name.
- **Skill** — a richer workflow that can fork context and restrict tools.

### Demo: write a forked, tool-restricted skill and a slash command, then read back the frontmatter

The cell writes `.claude/commands/review.md` (the `/review` command) and a skill at `.claude/skills/codebase-survey/SKILL.md` configured with `context: fork`, a restricted `allowed-tools` list (read-only — no `Write`, no `Bash`), and an `argument-hint`. It then parses and prints the skill's frontmatter as a settings table. Watch the `allowed-tools` line: `Write` and `Bash` are absent, so even if the skill's prompt asked to delete a file, the harness would block it. Try adding `"Bash"` to the list and re-running to see the table change.

In [ ]:
write_file(".claude/commands/review.md", """
---
description: Run the team's PR review checklist on the current diff
---
Review the staged changes against our standards:
1. Tests — does every bug fix include a failing-first regression test?
2. Types — is every public function signature type-hinted?
3. Commits — Conventional Commits format?
Report issues grouped by severity. Do not modify files.
""")

write_file(".claude/skills/codebase-survey/SKILL.md", """
---
name: codebase-survey
description: Summarize a package — entry points, modules, public API surface.
context: fork
allowed-tools: ["Read", "Grep", "Glob"]
argument-hint: <package path to survey, e.g. src/api>
---
# Codebase survey

Survey the package at the path provided.

1. Use Glob to list source files.
2. Use Grep to find entry points and exported names.
3. Use Read only to follow imports for the top-level modules.

Return a one-screen summary: entry points, key modules, public API surface.
Do not write or modify any file.
""")

print()
for skill_md in sorted(PROJECT.glob(".claude/skills/*/SKILL.md")):
    fm = parse_frontmatter(skill_md.read_text())
    print(f"skill: {fm.get('name')}")
    for key in ["context", "allowed-tools", "argument-hint"]:
        print(f"  {key:14s}: {fm.get(key)}")
    blocked = sorted({"Write", "Edit", "Bash"} - set(fm.get("allowed-tools", [])))
    print(f"  -> tools the skill CANNOT use : {blocked}")
    print(f"  -> verbose output isolated from main context: {fm.get('context') == 'fork'}")

> **🏫 During class:**
> 1. Run the cell. Read the printed skill settings aloud.
> 2. Say: "`context: fork` means this survey runs in a sub-agent — it can read 50 files and only a summary comes back. The `allowed-tools` list has no `Write` or `Bash`, so the skill physically cannot change or delete anything. That is enforced by the harness, not by hoping the prompt behaves."
> 3. Ask: "When would you *not* want `context: fork`?" (When you need the skill's output to stay in context for follow-up work — fork is for verbose, throwaway exploration.) Variation: add `"Bash"` to `allowed-tools` and re-run; watch the blocked-tools line shrink.

---
# 4. MCP servers — .mcp.json (project) and ~/.claude.json (personal)

MCP (Model Context Protocol) servers give Claude Code extra tools and resources. There are two scopes, mirroring the `CLAUDE.md` hierarchy:

| File | Scope | Committed? | Use for |
|---|---|---|---|
| `.mcp.json` (repo root) | Project | Yes | Shared team tooling — a GitHub server, the team's Jira server |
| `~/.claude.json` | User | No | Personal / experimental servers — your own scratch server |

At connection time Claude Code discovers tools from **all** configured servers, so project and personal servers are available **simultaneously**.

### Environment variable expansion

Never commit secrets. `.mcp.json` supports `${VAR}` expansion — the file references a variable name, the value comes from the environment:

```json
{ "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_TOKEN": "${GITHUB_TOKEN}" }
}}}
```

The committed file is safe — it holds the variable *name*, not the token.

### MCP resources

Beyond tools, servers expose **resources** — content catalogs (issue lists, doc trees, DB schemas). A resource lets the agent see what data exists without burning exploratory tool calls.

> **Safety note for this exercise:** we do **not** touch your real `~/.claude.json`. The cell writes a *simulated* copy inside the sandbox so the demo is reproducible and reversible.

### Demo: write project + personal MCP configs, expand credentials, list all servers

The cell writes a project `.mcp.json` (a `github` server using `${GITHUB_TOKEN}`) and a simulated personal config `home_claude.json` in the sandbox (an experimental `scratch` server). Then `expand_env` resolves `${VAR}` against a fake environment, and the cell merges both configs to print every server available in a session. Watch two things: the raw committed file still shows `${GITHUB_TOKEN}` (no secret on disk), and the merged list contains servers from **both** scopes. Remove `GITHUB_TOKEN` from `fake_env` to see expansion leave the placeholder unresolved.

In [ ]:
import string

write_file(".mcp.json", """
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_TOKEN": "${GITHUB_TOKEN}" }
    }
  }
}
""")

# Simulated ~/.claude.json — written into the sandbox, NOT your real home dir.
write_file("home_claude.json", """
{
  "mcpServers": {
    "scratch": {
      "command": "uv",
      "args": ["run", "python", "/Users/me/experiments/scratch_server.py"]
    }
  }
}
""")

# A fake environment — stands in for the shell that launches Claude Code.
fake_env = {"GITHUB_TOKEN": "ghp_example_not_a_real_token"}


def expand_env(obj, env):
    """Recursively expand ${VAR} placeholders; leave unknown vars untouched."""
    if isinstance(obj, dict):
        return {k: expand_env(v, env) for k, v in obj.items()}
    if isinstance(obj, list):
        return [expand_env(v, env) for v in obj]
    if isinstance(obj, str):
        return string.Template(obj).safe_substitute(env)
    return obj


project_cfg = json.loads((PROJECT / ".mcp.json").read_text())
personal_cfg = json.loads((PROJECT / "home_claude.json").read_text())

print("Raw .mcp.json env block (committed to git — no secret here):")
print("  ", project_cfg["mcpServers"]["github"]["env"])

resolved = expand_env(project_cfg, fake_env)
print("After ${...} expansion at launch:")
print("  ", resolved["mcpServers"]["github"]["env"])

print("\nServers available simultaneously in one session:")
all_servers = {**project_cfg["mcpServers"], **personal_cfg["mcpServers"]}
for name, spec in all_servers.items():
    scope = "project  (.mcp.json)" if name in project_cfg["mcpServers"] else "personal (~/.claude.json)"
    print(f"  {name:9s} [{scope}]  -> {spec['command']} {' '.join(spec['args'])}")

> **🏫 During class:**
> 1. Run the cell. Compare the two printed `env` blocks — same key, but the raw one shows `${GITHUB_TOKEN}` and the resolved one shows the value.
> 2. Say: "The committed file never contains the secret — it names an environment variable. The token only exists in the shell that launches Claude Code. That is how a team shares one `.mcp.json` in git without leaking credentials."
> 3. Ask: "Both a project and a personal server are listed — which tools can the agent use?" (Both — every configured server's tools are discovered at connection time.) Variation: delete `GITHUB_TOKEN` from `fake_env` and re-run; the placeholder stays unresolved — exactly the bug you would debug if a teammate forgot to export it.

---
# 5. Plan mode vs direct execution

Claude Code can **plan before it acts**. Plan mode explores the codebase and proposes an approach for approval *before* editing anything. Direct execution just makes the change.

### When each fits

| Signal | Mode |
|---|---|
| One file, clear scope, obvious fix (a stack trace points right at it) | Direct execution |
| Multiple valid approaches; an architectural decision | Plan mode |
| Large-scale / multi-file change (e.g. a migration touching 45+ files) | Plan mode |
| Need to explore unfamiliar code before committing | Plan mode |

Plan mode's value is **avoiding costly rework** — you catch a wrong approach in a plan review instead of after 40 files are already edited.

### The three exercise tasks
1. **Single-file bug fix** with a clear stack trace — the scope is obvious.
2. **Multi-file library migration** — one approach, but 45+ files; far-reaching.
3. **New feature with several valid designs** — the design decision is the hard part.

### The Explore subagent

For a verbose discovery phase, the **Explore subagent** isolates that output and returns only a summary — it keeps the main context from filling with file dumps during a multi-phase task. "Plan mode for investigation, direct execution for the now-understood implementation" is a common, effective combination.

### Demo: score each task and recommend a mode

`recommend(task)` scores a task on four factors — files touched, number of viable approaches, architectural impact, and code familiarity — and returns a recommendation with reasons. The cell runs it on the three exercise tasks. Watch the single-file bug fix come back "direct execution" while the migration and the open-ended feature both come back "plan mode" — but for different reasons (breadth vs design ambiguity). Edit a task's `files` count or `approaches` to see the recommendation flip.

In [ ]:
tasks = [
    {"name": "Fix off-by-one in pagination (clear stack trace)",
     "files": 1, "approaches": 1, "architectural": False, "familiar": True},
    {"name": "Migrate requests -> httpx across the codebase",
     "files": 47, "approaches": 1, "architectural": False, "familiar": True},
    {"name": "Add a notifications feature (email? webhook? in-app?)",
     "files": 8, "approaches": 3, "architectural": True, "familiar": False},
]


def recommend(task):
    """Recommend plan mode vs direct execution, with reasons."""
    reasons = []
    if task["files"] >= 5:
        reasons.append(f"touches {task['files']} files (large-scale)")
    if task["approaches"] >= 2:
        reasons.append(f"{task['approaches']} valid approaches to weigh")
    if task["architectural"]:
        reasons.append("has architectural implications")
    if not task["familiar"]:
        reasons.append("needs codebase exploration first")
    mode = "PLAN MODE" if reasons else "DIRECT EXECUTION"
    if not reasons:
        reasons = ["single file, one obvious approach, clear scope"]
    return mode, reasons


for task in tasks:
    mode, reasons = recommend(task)
    print(task["name"])
    print(f"  -> {mode}")
    for r in reasons:
        print(f"     - {r}")
    print()

> **🏫 During class:**
> 1. Run the cell. Take the three tasks in order.
> 2. Say: "Plan mode is not 'careful mode' for everything — it earns its cost when a wrong move is expensive. The bug fix has one right answer, so planning is overhead. The migration is one approach but 47 files — you plan to sequence it. The feature has three designs — you plan because the *decision* is the work."
> 3. Ask: "The migration has only one approach — why still plan?" (Scale: 47 files means you want an agreed sequence and checkpoints before touching anything.) Variation: set the bug fix's `files` to 6 and re-run — watch it flip to plan mode.

---
# 6. Recap + practice exercises

### Recap (run through these out loud)
- **CLAUDE.md hierarchy** — user (personal, not shared) -> project (committed, the team contract) -> directory (local additions). Shared standards belong at the project level; `/memory` shows what loaded.
- **.claude/rules/** — topic files with `paths:` globs load *conditionally*; globs beat a directory `CLAUDE.md` for file types spread across the tree.
- **Skills & commands** — `.claude/commands/` for slash commands, `.claude/skills/` for workflows; `context: fork` isolates verbose output, `allowed-tools` restricts what a skill can touch, `argument-hint` prompts for input.
- **MCP config** — `.mcp.json` (project, committed, `${VAR}` for secrets) and `~/.claude.json` (personal); all servers' tools are available at once.
- **Plan mode** — for multi-file, multi-approach, or architectural work; direct execution for well-scoped single changes.

### Exercises (progressively harder)
1. Add a `deployment.md` rule scoped to `paths: ["**/Dockerfile", "**/*.yaml"]` and extend the `rules_for` test list to prove it loads for `k8s/prod.yaml` but not `README.md`.
2. Add a directory-level `CLAUDE.md` under `src/ui/` and write a function that, given a file path, lists every `CLAUDE.md` that applies (root + matching directories) in load order.
3. Write a second skill `release-notes` *without* `context: fork`, and add a column to the skill table showing whether each skill's output pollutes the main context.
4. Extend `expand_env` to *report* any `${VAR}` left unresolved, and make a `.mcp.json` loader fail loudly when a required credential is missing from the environment.
5. Give `recommend()` a fifth factor — "reversible?" — and re-rank the three tasks; discuss whether reversibility should outweigh file count.

In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# Uncomment and complete:

# write_file(".claude/rules/deployment.md", """
# ---
# paths: ["**/Dockerfile", "**/*.yaml"]
# ---
# # Deployment conventions
# - Pin base image digests; never deploy from :latest.
# - One service per manifest.
# """)
#
# for f in ["k8s/prod.yaml", "Dockerfile", "src/api/users.py", "README.md"]:
#     print(f, "->", rules_for(f))